# Computational lab 1 — Universal approximation, in numbers**M2 MACIA · Physics-Informed Neural Networks for PDEs**  Paul Boureau — AGM, CY Cergy Paris UniversityThis notebook is the companion to the problem class *Session 1 · Universal approximation*.Each part reproduces, numerically, a statement you proved on paper. Nothing here replaces a proof:the point is to see the objects, and to meet — once, concretely — the gap between *a good networkexists* and *an algorithm finds it*.Cells marked **`# TODO`** are yours. Everything else is given.

In [ ]:
import torch, numpy as np, matplotlib.pyplot as plttorch.set_default_dtype(torch.float64)      # double precision: we will measure small errorstorch.manual_seed(0); np.random.seed(0)TEAL, WINE, OCHRE = "#0E6A62", "#8A3A55", "#7E5C0E"print("torch", torch.__version__)

---## Part 1 — Slabs, not bumps*Companion to Exercise 1 of the problem class, and to Proposition 1.5 of the lecture notes.*A one-hidden-layer network is$$\Phi_\theta(x) \;=\; \sum_{i=1}^n c_i\,\sigma(w_i\cdot x + b_i),\qquad \theta = (c, W, b).$$We write it by hand, with tensors, so that nothing is hidden. `W` has shape `(n, d)`.

In [ ]:
def phi(x, c, W, b, act=torch.tanh):    """x: (M, d) -> (M,).  One hidden layer, n neurons, written out explicitly."""    return act(x @ W.T + b) @ cd, n = 2, 3x = torch.randn(5, d); c = torch.randn(n); W = torch.randn(n, d); b = torch.randn(n)print(phi(x, c, W, b).shape)

### 1.1 — The bump in dimension 1 (Exercise 1, item 3)You showed that for sigmoidal $\sigma$ and $a<b$,$\sigma(\lambda(x-a)) - \sigma(\lambda(x-b)) \to \mathbf{1}_{(a,b)}$ pointwise off $\{a,b\}$.Watch it happen.

In [ ]:
xs = torch.linspace(-1, 2, 1000)sig = lambda t: torch.sigmoid(t)                 # a sigmoidal activation: limits 0 and 1a, bb = 0.3, 0.8plt.figure(figsize=(6, 2.6))for lam, col in zip([2, 10, 100], [OCHRE, WINE, TEAL]):    plt.plot(xs, sig(lam*(xs-a)) - sig(lam*(xs-bb)), color=col, lw=1.4, label=f"$\\lambda={lam}$")plt.legend(frameon=False); plt.xlabel("$x$"); plt.title("two sigmoids make a bump")plt.tight_layout(); plt.show()

### 1.2 — In dimension 2 it is a slab, not a bump (Exercise 1, item 5) — **TODO**Plot the level sets of $\sigma(w_1\cdot x)$, of $\sigma(w_2\cdot x)$ with $w_1=(1,0)$, $w_2=(0,1)$,and of their half-sum, on $[-2,2]^2$.Look at the level sets, and answer in one sentence: *why can no finite linear combination of ridgefunctions be the indicator of a square?*

In [ ]:
g = torch.linspace(-2, 2, 300)X, Y = torch.meshgrid(g, g, indexing="ij")P = torch.stack([X.reshape(-1), Y.reshape(-1)], 1)      # (M, 2) grid points# TODO: evaluate the three functions on P with phi(...), reshape to X.shape,#       and draw them side by side with plt.contour / plt.contourf.

### 1.3 — Proposition 1.5, measured (Exercise 1 again, and §1 of the notes) — **TODO**You proved that if $f\in\Sigma_1(\sigma)$ then $\nabla f(x)$ is collinear to a *fixed* vector $w$for every $x$, and that this fails for $\frac12(f_1+f_2)$. Measure it: sample points at random,compute $\nabla f$ with `torch.autograd.grad`, normalise, and report the angular spread of thedirections.*(You are using automatic differentiation two weeks early. Session 2 explains what it does; for now,`autograd.grad` returns the exact gradient, not a finite difference.)*

In [ ]:
def angular_spread(fun, n_pts=4000):    X = (torch.rand(n_pts, 2)*4 - 2).requires_grad_(True)    # TODO: y = fun(X); get g = dy/dX with torch.autograd.grad(y.sum(), X)    #       normalise each row, take the angle atan2(g1, g0), return max - min    raise NotImplementedErrorprint("single ridge :", angular_spread(lambda X: torch.tanh(X @ torch.tensor([1.0, 0.0]))))print("half-sum     :", angular_spread(lambda X: 0.5*(torch.tanh(X @ torch.tensor([1.0, 0.0]))                                                    + torch.tanh(X @ torch.tensor([0.0, 1.0])))))

---## Part 2 — A polynomial activation is not rescued by width, and *is* helped by depth*Companion to Exercise 2, items 5 and 6.*You proved: with $\sigma(t)=t^2$ and depth $L$, the realised function is a polynomial of degree atmost $2^{L-1}$ — whatever the width. And that letting $L$ grow recovers all polynomials.Here is the cheapest possible test, and it needs no training at all: take a **random** deepquadratic network, sample it, and fit a polynomial. If the prediction is right, degree $2^{L-1}$should fit to machine precision and degree $2^{L-1}-1$ should not.

In [ ]:
def deep_quadratic(x, L, width=6, seed=0):    """Random network, activation t^2, depth L (so L-1 hidden layers). x: (M,) -> (M,)."""    torch.manual_seed(seed)    h = x.reshape(-1, 1)    for _ in range(L - 1):        W = torch.randn(h.shape[1], width); b = torch.randn(width)        h = (h @ W + b)**2    W = torch.randn(h.shape[1], 1); b = torch.randn(1)    return (h @ W + b).squeeze()xs = torch.linspace(-1, 1, 400)# TODO: for L = 2, 3, 4, evaluate the network, then np.polyfit at degrees 2**(L-1)-1 and 2**(L-1),#       and print the relative sup residual of each fit. What do you predict before running it?

---## Part 3 — Existence is not an algorithm*Companion to Exercise 5, items 2 and 3, and to Exercise 2.4 of the lecture notes.*This is the part that matters for the rest of the course.

### 3.1 — The loss is not convex: the two-line counterexample, drawn — **TODO**In Exercise 2.4 of the lecture notes you checked that $\theta_+=(c,w,b)=(1,1,0)$ and$\theta_-=(-1,-1,0)$ both realise $\tanh$, so $J(\theta_+)=J(\theta_-)=0$ for the target$f=\tanh$, while the midpoint realises the zero function.Plot $t \mapsto J\big((1-t)\theta_+ + t\theta_-\big)$ on $[0,1]$.

In [ ]:
xg = torch.linspace(-1, 1, 2001); f = torch.tanh(xg)def J(theta):    c, w, b = theta    return ((c*torch.tanh(w*xg + b) - f)**2).mean().item()tp = torch.tensor([1., 1., 0.]); tm = torch.tensor([-1., -1., 0.])# TODO: evaluate J along the segment and plot it. What is the value at t = 1/2, and why?

### 3.2 — $\mathcal{E}_{\mathrm{app}}$ against $\mathcal{E}_{\mathrm{opt}}$ — **TODO**Cybenko's theorem says a good $\theta$ exists. Here we separate *existence* from *finding*, with atrick: if the inner parameters $(w_i,b_i)$ are **frozen at random values**, then $J$ depends on theouter coefficients $c$ only, and $\min_c J$ is a **linear least-squares problem** — convex, solvedexactly by `torch.linalg.lstsq`. That gives an achievable error with no optimisation difficulty atall.Then train *all* the parameters by gradient descent, with the same $n$, and compare.Target: $f(x) = \sin(2\pi x) + \tfrac14\sin(16\pi x)$ on $[0,1]$ — one slow mode and one fast one.

In [ ]:
def target(x): return torch.sin(2*np.pi*x) + 0.25*torch.sin(16*np.pi*x)xtr = torch.linspace(0, 1, 2000); ytr = target(xtr)def features(s, u):                      # tanh(s_i (x - u_i)) : slope s_i, transition at u_i    return torch.tanh(s*(xtr.reshape(-1, 1) - u))def random_features(n, seed):    """Freeze (s, u) at random, solve for c by least squares. Return the RMS error."""    g = torch.Generator().manual_seed(seed)    s = torch.randn(n, generator=g)*20.0; u = torch.rand(n, generator=g)    # TODO: build A = [features(s,u), 1], solve with torch.linalg.lstsq, return the RMS error    raise NotImplementedErrordef train_everything(n, seed, steps=4000, lr=5e-3):    """Same architecture, but optimise s, u, c, b0 jointly with Adam."""    g = torch.Generator().manual_seed(seed)    s  = (torch.randn(n, generator=g)*20.0).requires_grad_(True)    u  = (torch.rand(n, generator=g)).requires_grad_(True)    c  = (torch.randn(n, generator=g)/np.sqrt(n)).requires_grad_(True)    b0 = torch.zeros(1, requires_grad=True)    opt = torch.optim.Adam([s, u, c, b0], lr=lr)    # TODO: the training loop. At each step: zero the gradients, compute the mean squared error    #       between tanh(s*(x-u)) @ c + b0 and ytr, call .backward(), call opt.step().    raise NotImplementedError# TODO: loop over n = 8, 16, 32, 64, 128, 256 and tabulate the two errors. Predict first.

---## Part 4 (take home) — A finite-dimensional shadow of "discriminatory"*Companion to Exercise 4 and Definition 1 of the problem class.*The discriminatory property says: the only finite signed measure $\mu$ with$\int \sigma(wx+b)\,d\mu = 0$ for **all** $(w,b)$ is $\mu = 0$. Replace $K$ by $M$ sample pointsand the measure by a weight vector $\mu\in\mathbb{R}^M$. The condition becomes$A^{\!\top}\mu = 0$, where $A_{ij} = \sigma(w_j x_i + b_j)$.So *"$\sigma$ is discriminatory"* becomes, in finite dimension: **$A$ has full row rank $M$ onceenough features are drawn**. And a non-discriminatory $\sigma$ should show a rank that saturatesbelow $M$ — the surviving $\mu$'s spanning the annihilator of the whole class.Compute the numerical rank of $A$ as $n$ grows, for $\sigma=\tanh$ and for $\sigma(t)=t^2$.

In [ ]:
M = 40; xs4 = torch.linspace(-1, 1, M)# TODO: for each activation and each n in [2,5,10,20,40,100,400], build A (M x n) from random#       (w, b) and print torch.linalg.matrix_rank(A, rtol=1e-10).#       Predict the saturation value for t^2 before running.

---## What to take away1. Ridge functions are **slabs**. In $d\ge2$ nothing is localised, and the one-dimensional intuition   of a staircase of bumps does not transpose. This is why the proof needs Hahn–Banach and Fourier.2. A **polynomial activation** is capped by width and freed by depth, exactly as predicted, and the   cap is visible to machine precision without training anything.3. The loss is **not convex**, and the counterexample is not pathological: it comes from the   parametrisation being many-to-one on functions.4. **Existence is not an algorithm.** On the same class, a convex solve reaches $10^{-10}$ where   gradient training stalls at $10^{-2}$. Sessions 4 and 6 are about that gap.*Next session: how `autograd.grad` actually computed those derivatives, why it is exact where afinite difference is not, and what can and cannot be proved about the optimiser you just used.*